# AB-RAG Phase 1 on Kaggle
Full traces from the local model (token-probability S1) for both datasets, then the offline policy replays (`10_policies.py`) and the N2–N4 report (`12_report.py`).

**Once:** verify your phone number on Kaggle (this unlocks GPU and internet). On your machine run `python notebooks/make_code_zip.py` and upload the zip as a **private** dataset (Datasets → New Dataset).

**Every session:**
1. Add Input → your code dataset. From the second session on, also add this notebook's previous output, so the run resumes where it stopped.
2. Settings → Accelerator **GPU T4 x2**, Internet **on**.
3. Save Version → **Save & Run All**. An interactive session ends after 20 idle minutes; a saved version runs unattended.

Each question is written as soon as it is answered, and the run stops on its own after `MAX_HOURS`, so a version always ends inside Kaggle's 12-hour limit and keeps its output.

In [ ]:
import time
SESSION_START = time.time()

DATASETS = ["hotpotqa", "triviaqa"]   # on T4 x2 each dataset gets its own GPU and they run in parallel
MODEL = None        # None = the config's Qwen/Qwen2.5-1.5B-Instruct; e.g. "Qwen/Qwen2.5-3B-Instruct"
DEVICE_MAP = None   # "auto" splits one model over both T4s (7-8B in fp16); datasets then run one after another
LOAD_4BIT = False   # 4-bit weights for 7-8B on one T4; changes S1, so traces go to their own -4bit file
N_SAMPLES = 5       # sampled answers per round
MAX_HOURS = 10.5    # stop answering after this many hours from session start (Kaggle's limit is 12)

In [ ]:
import glob, os, shutil, subprocess, sys

found = sorted(glob.glob("/kaggle/input/**/configs/config.py", recursive=True))
if not found:   # the upload was kept as a zip: unpack it into scratch space
    zips = sorted(glob.glob("/kaggle/input/**/abrag_code*.zip", recursive=True))
    assert zips, "add the code dataset as an input (Add Input -> Your Work -> Datasets)"
    shutil.unpack_archive(zips[0], "/tmp/abrag_src")
    found = ["/tmp/abrag_src/configs/config.py"]
CODE = os.path.dirname(os.path.dirname(found[0]))
WORK, OUT = "/tmp/abrag", "/kaggle/working/results"   # code in scratch space; only results/ is saved
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(CODE, WORK, ignore=shutil.ignore_patterns("results"))
os.makedirs(OUT, exist_ok=True)
os.symlink(OUT, f"{WORK}/results")
os.chdir(WORK)
sys.path.insert(0, WORK)
print("code from", CODE, f"({len(found)} code inputs found, using the first)" if len(found) > 1 else "")

def n_lines(path):
    return sum(1 for _ in open(path, encoding="utf-8")) if os.path.exists(path) else 0

# cached retrieval rankings from the zip, so this session skips re-ranking
for p in glob.glob(f"{CODE}/results/*"):
    if not os.path.exists(f"{OUT}/{os.path.basename(p)}"):
        shutil.copy(p, OUT)
# resume: earlier versions' outputs, added as inputs, hold results/traces_*.jsonl
for p in glob.glob("/kaggle/input/**/traces_*.jsonl", recursive=True):
    dst = f"{OUT}/{os.path.basename(p)}"
    if n_lines(p) > n_lines(dst):
        shutil.copy(p, dst)
        print(f"resuming {os.path.basename(p)}: {n_lines(dst)} questions already answered")

In [ ]:
def ensure(module, package):
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

for module, package in [("rank_bm25", "rank-bm25"), ("faiss", "faiss-cpu"),
                        ("sentence_transformers", "sentence-transformers")]:
    ensure(module, package)
if LOAD_4BIT:
    ensure("bitsandbytes", "bitsandbytes")

import torch
N_GPU = torch.cuda.device_count()
assert N_GPU, "no GPU: Settings -> Accelerator -> GPU T4 x2"
for i in range(N_GPU):
    print(i, torch.cuda.get_device_name(i))
try:   # newer PyTorch builds can lack kernels for older GPUs such as the P100
    torch.ones(1, device="cuda").add_(1)
except RuntimeError as e:
    raise SystemExit(f"this GPU does not work with the installed PyTorch: {e}\n"
                     "switch Settings -> Accelerator to GPU T4 x2")

In [ ]:
# download the models once up front: fails fast if Internet is off, and parallel runs share the cache
from huggingface_hub import snapshot_download
from configs.config import CONFIG
for m in (CONFIG["embed_model"], MODEL or CONFIG["local_model"]):
    snapshot_download(m, allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors"])
    print("ready:", m)

In [ ]:
def traces_cmd(dataset, hours):
    cmd = [sys.executable, "experiments/09_full_traces.py", "--dataset", dataset, "--split", "all",
           "--backend", "local", "--n_samples", str(N_SAMPLES), "--max_hours", f"{hours:.2f}"]
    if MODEL:
        cmd += ["--model", MODEL]
    if DEVICE_MAP:
        cmd += ["--device_map", DEVICE_MAP]
    if LOAD_4BIT:
        cmd += ["--load_4bit"]
    return cmd

def hours_left():
    return MAX_HOURS - (time.time() - SESSION_START) / 3600

def progress():
    return {os.path.basename(p): n_lines(p) for p in sorted(glob.glob(f"{OUT}/traces_*.jsonl"))}

if N_GPU >= len(DATASETS) and not DEVICE_MAP:      # one dataset per GPU, in parallel
    procs = [subprocess.Popen(traces_cmd(ds, hours_left()), stdout=open(f"{OUT}/log_{ds}.txt", "a"),
                              stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i)})
             for i, ds in enumerate(DATASETS)]
    last = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(30)
        if time.time() - last > 600:
            print(f"{(time.time() - SESSION_START) / 3600:.1f} h", progress(), flush=True)
            last = time.time()
else:                                              # one after another, sharing the time budget
    for ds in DATASETS:
        if hours_left() > 0.1:
            subprocess.run(traces_cmd(ds, hours_left()), stdout=open(f"{OUT}/log_{ds}.txt", "a"),
                           stderr=subprocess.STDOUT)

for ds in DATASETS:   # end of each log (progress bars, usage, errors)
    log = open(f"{OUT}/log_{ds}.txt", encoding="utf-8", errors="replace").read().replace("\r", "\n")
    print(f"--- {ds}\n" + "\n".join(l for l in log.splitlines() if l.strip())[-1500:])
print(progress())

In [ ]:
# offline replays and the report (CPU, seconds); N1 (11_value_policy.py) is set aside
def run(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout[-6000:] + (r.stderr[-2000:] if r.returncode else ""))

for p in sorted(glob.glob(f"{OUT}/traces_*.jsonl")):
    dataset, label = os.path.basename(p)[len("traces_"):-len(".jsonl")].split("_", 1)
    print(f"\n===== {dataset} / {label}: {n_lines(p)} questions")
    for split in ("test", "dev"):
        run("experiments/10_policies.py", "--dataset", dataset, "--backend", label, "--split", split)
run("experiments/12_report.py")

**Next session:** add this notebook's latest output as an input and Save & Run All again; answered questions are skipped. A dataset is complete at 800 questions (dev 300 + test 500). Download `results/` from the Output tab into your local `results/` folder to analyse it on your machine.